# NYC HVFHV — Data Preprocessing

Applies data quality filters and type conversions to the raw HVFHV trip data 
(see `01_data_understanding.ipynb` for the EDA and rationale behind each step), 
producing a clean table (`fhvhv_clean`) for downstream hypothesis testing.

**Input:** `fhvhv_raw` (materialized from `../data/raw/fhvhv_tripdata_2026-*.parquet`)
**Output:** `fhvhv_clean`

## Preprocessing Steps Applied
- Removed `trip_miles <= 0` (~11.5K rows, 0.01%)
- Removed invalid trip duration — `dropoff_datetime <= pickup_datetime` (2 rows)
- Removed `base_passenger_fare <= 0` (0.14%)
- Removed `driver_pay <= 0` (0.06%)
- Removed timestamp discrepancies — `on_scene_datetime` before `request_datetime` (~1.9M rows, 1.8%) 
  and `on_scene_datetime` after `pickup_datetime` (~3.2K rows, 0.003%)
- Converted Y/N flags to booleans — `shared_request_flag`, `shared_match_flag`, 
  `access_a_ride_flag`, `wav_request_flag`, `wav_match_flag`
- Mapped provider codes to names — HV0003 → Uber, HV0005 → Lyft

In [1]:
import duckdb

con = duckdb.connect("../data/nyc_fhvhv_2026.duckdb")

# Our source of truth is the raw fhvhv data
con.execute("""
CREATE OR REPLACE TABLE fhvhv_raw AS
SELECT DISTINCT *
FROM read_parquet('../data/raw/fhvhv_tripdata_2026-*.parquet');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
con.sql("""
SELECT COUNT(*) FROM fhvhv_raw;
""").df()

,count_star()
0,105996113


In [3]:
# Create a cleaned version of the fhvhv data
con.execute("""
CREATE OR REPLACE TABLE fhvhv_clean AS
SELECT
    CASE 
        WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
        WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
        ELSE hvfhs_license_num
    END AS provider_name,
    dispatching_base_num,
    originating_base_num,
    request_datetime,
    on_scene_datetime,
    pickup_datetime,
    dropoff_datetime,
    PULocationID,
    DOLocationID,
    trip_miles,
    trip_time,
    base_passenger_fare,
    tolls,
    bcf,
    sales_tax,
    congestion_surcharge,
    airport_fee,
    tips,
    driver_pay,
    (shared_request_flag = 'Y') AS shared_request_flag,
    (shared_match_flag = 'Y')   AS shared_match_flag,
    (access_a_ride_flag = 'Y')  AS access_a_ride_flag,
    (wav_request_flag = 'Y')    AS wav_request_flag,
    (wav_match_flag = 'Y')      AS wav_match_flag,
    cbd_congestion_fee
FROM fhvhv_raw
WHERE trip_miles > 0
    AND dropoff_datetime > pickup_datetime
    AND base_passenger_fare > 0
    AND driver_pay > 0
    AND on_scene_datetime >= request_datetime
    AND on_scene_datetime <= pickup_datetime;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
con.sql("""
SELECT
    *
FROM fhvhv_clean
LIMIT 5;
""").df()

,provider_name,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,cbd_congestion_fee
0,Uber,B03404,B03404,2026-01-12 12:53:58,2026-01-12 13:03:04,2026-01-12 13:03:06,2026-01-12 13:13:04,35,72,1.53,...,0.0,0.0,0.0,8.47,False,False,False,False,False,0.0
1,Uber,B03404,B03404,2026-01-12 13:49:54,2026-01-12 13:54:03,2026-01-12 13:55:50,2026-01-12 14:17:16,112,80,3.76,...,0.0,0.0,0.0,29.03,False,False,False,False,False,0.0
2,Uber,B03404,B03404,2026-01-12 12:51:51,2026-01-12 13:00:19,2026-01-12 13:02:19,2026-01-12 13:14:29,225,36,1.60,...,0.0,0.0,0.0,10.01,False,False,False,False,False,0.0
3,Uber,B03404,B03404,2026-01-12 13:55:38,2026-01-12 13:56:54,2026-01-12 13:58:54,2026-01-12 14:21:39,157,146,3.50,...,0.0,0.0,0.0,19.33,False,False,False,False,False,0.0
4,Uber,B03404,B03404,2026-01-12 13:44:59,2026-01-12 13:46:09,2026-01-12 13:47:26,2026-01-12 13:57:18,116,42,0.64,...,0.0,0.0,0.0,7.39,False,False,False,False,False,0.0


In [5]:
con.sql("""
DESCRIBE fhvhv_clean;
""").df()

,column_name,column_type,null,key,default,extra
0,provider_name,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,trip_miles,DOUBLE,YES,None,None,None


In [6]:
con.sql("""
SELECT COUNT(*) FROM fhvhv_clean;
""").df()

,count_star()
0,103921454


In [7]:
con.sql("""
SELECT
    SUM(trip_miles <= 0) AS invalid_trip_miles,
    SUM(dropoff_datetime <= pickup_datetime) AS invalid_duration,
    SUM(base_passenger_fare <=0) AS invalid_fare,
    SUM(driver_pay <=0) AS invalid_driver_pay,
    SUM(on_scene_datetime < request_datetime) AS invalid_request_time,
    SUM(on_scene_datetime > pickup_datetime) AS invalid_pickup_time
FROM fhvhv_clean;
""").df()

,invalid_trip_miles,invalid_duration,invalid_fare,invalid_driver_pay,invalid_request_time,invalid_pickup_time
0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
con.sql("""
SELECT 
    provider_name,
    COUNT(*) AS total_trips,
    SUM(CASE WHEN originating_base_num IS NULL THEN 1 ELSE 0 END) AS nulls
FROM fhvhv_clean
GROUP BY provider_name;
""").df()

,provider_name,total_trips,nulls
0,Lyft,29310514,29266702.0
1,Uber,74610940,0.0


In [9]:
con.sql("""
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) AS invalid_base_passenger_fare_count,
    SUM(CASE WHEN driver_pay <= 0 THEN 1 ELSE 0 END) AS invalid_driver_pay_count,
    (SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS invalid_base_passenger_fare_pct,
    (SUM(CASE WHEN driver_pay <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS invalid_driver_pay_pct
FROM fhvhv_clean;
""").df()

,total_trips,invalid_base_passenger_fare_count,invalid_driver_pay_count,invalid_base_passenger_fare_pct,invalid_driver_pay_pct
0,103921454,0.0,0.0,0.0,0.0


In [10]:
con.close()